# Temporal-v3 — парный пилот без обучения

**Не нажимай Run all. Выполняй кодовые ячейки 1–7 отдельно.** Это exploratory-проверка дополнительной информации, не улучшение архитектуры при одинаковом входе. Test и holdout не читаются. Никакая ячейка не обучает модель.


## 1. Drive и пути

Новый каталог `temporal_v3_paired_pilot/v1` не связан с frozen runs. Если повторяешь пилот после изменения кода/входов, выбери новый суффикс вместо v1; старые результаты автоматически не перезаписываются.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/diffusion-sources')
DATA_DIR = DRIVE_ROOT / 'data/facebook_main'
RUN_ROOT = DRIVE_ROOT / 'reports/runs/facebook_main_v2'
RUN_DIR = RUN_ROOT / 's1b/seed_7026'
PILOT_DIR = DRIVE_ROOT / 'reports/runs/temporal_v3_paired_pilot/v1'
REPO = Path('/content/diffusion-sources')
REPOSITORY = 'https://github.com/1habibi/diffusion-sources-localization-.git'
DEVICE = 'cuda'
print('New pilot output:', PILOT_DIR)

## 2. Код, установка и проверка импорта в ядре

Перед этим изменения v3 должны быть опубликованы в репозитории. Ошибка о недостающем temporal_pilot_cli означает, что checkout ещё не содержит новый код. После сброса runtime повторяются ячейки 1–3.

In [ ]:
import sys, subprocess
from pathlib import Path
print('[setup 1/4] Получение кода', flush=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
print('[setup 2/4] Зависимости', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'networkx', 'numpy', 'scipy', 'matplotlib', 'PyYAML', 'scikit-learn',
    'streamlit', 'tqdm', 'torch-geometric', 'pytest'], check=True)
print('[setup 3/4] Установка проекта', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), '--no-deps'], check=True)
sys.path.insert(0, str(REPO / 'src'))
import diffusion_sources
from diffusion_sources import temporal_pilot_cli
assert REPO.resolve() in Path(diffusion_sources.__file__).resolve().parents, 'Импортирован другой checkout'
print('Kernel Python:', sys.executable)
print('Project import:', diffusion_sources.__file__)
import torch
DEVICE = globals().get('DEVICE', 'cuda')
if DEVICE == 'cuda':
    assert torch.cuda.is_available(), 'Выбери Runtime -> Change runtime type -> GPU'
    print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'device:', DEVICE)
subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)
print('[setup 4/4] Готово: kernel импортирует код v3', flush=True)

## 3. Пути и команда запуска

Проверяются только development-входы и frozen checkpoint. Validation читается позже в своей ячейке. Ранний снимок — отдельная неполная выборка после шага 1, **без принудительного включения источников**; наблюдения не являются накопительным журналом.

In [ ]:
import sys, subprocess, json
for root, names in [(DATA_DIR, ['graph.npz', 'config.yaml', 'train.npz']),
                    (RUN_DIR, ['config.yaml', 'best_model.pt', 'metrics.json'])]:
    for name in names:
        path = root / name
        assert path.is_file(), f'Не найден файл: {path}'
        print('Input:', path)
def run_stage(stage, repeat_runs=()):
    command = [sys.executable, '-u', '-m', 'diffusion_sources.temporal_pilot_cli',
        stage, '--data-dir', str(DATA_DIR), '--run-dir', str(RUN_DIR),
        '--output-dir', str(PILOT_DIR), '--device', DEVICE]
    for run in repeat_runs:
        command += ['--repeat-run', str(run)]
    subprocess.run(command, cwd=REPO, check=True)
print('Пути готовы; следующая ячейка — smoke.')

## 4. Smoke — 6 train-примеров

Проверка replay, совпадения beta=0 и времени выполнения. При ошибке остановись и пришли вывод. Обучения нет.

In [ ]:
run_stage('smoke')

## 5. Выбор поправки — только 540 train-примеров

Сетка beta: 0 / 0.1 / 0.25 / 0.5 / 1.0. Выбор сохраняется в manifest, validation для настройки не используется. **Не запускай эту ячейку, если smoke не завершился успешно.**

In [ ]:
run_stage('select')

## 6. Парная оценка — все 1998 validation-примеров

Один beta из предыдущей ячейки; никакого подбора по validation. F1 контроля должен совпасть с сохранённым S1b. Gate: delta >= 0.02, CI lower > 0 и без существенного ухудшения k=2/3. При gate=false следующий этап не запускается.

In [ ]:
run_stage('validate')

## 7. Остальные frozen seed — только после primary pass

Те же early masks и beta для 7027/7028, только inference. После результата отправь отчёт для решения о дальнейшей работе. Положительный пилот ещё не разрешает автоматически запускать обучение или открывать holdout.

In [ ]:
import json
primary = json.loads((PILOT_DIR / 'validation/metrics.json').read_text())
if not primary['gate']['passed']:
    print('Primary gate не пройден. Не запускаем frozen repeats или обучение.')
else:
    run_stage('confirm', [RUN_ROOT / 'frozen_candidate/seed_7027',
                          RUN_ROOT / 'frozen_candidate/seed_7028'])

## 8. Ранний baseline: подготовка отдельного анализа

Продолжение после завершённого пилота. Старые ячейки 4–7 **не повторять**. После обновления кода их строгая code identity отличается; для нового анализа старые отчёты проверяются отдельно. Если runtime сброшен, сначала подключи Drive и установи проект ячейками 1–3, затем переходи сюда. Новые ячейки 8–12 не обучают и не меняют beta.

Сравниваем snapshot S1b, ранний снимок без learned scores и S1b + early. Count-head во всех вариантах одна и та же: это абляция ранжирования, не полностью безнейросетевой алгоритм. При ties считаем точный ожидаемый F1 равномерного выбора, не используем ID или scores как tie-break. Отчёт содержит F1, разрезы и paired CI; геометрические метрики для ожидаемого baseline не заявляются.

Вывод новых стадий идёт прямо из notebook kernel. Все результаты в отдельной папке temporal_v3_early_baseline/v1. Если code/input hashes изменились после начала анализа, выбери новый suffix.

In [ ]:
import subprocess, json, torch, importlib
print('[baseline setup] Обновление кода без повторного пилота...', flush=True)
subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
import diffusion_sources.temporal_early_baseline as _early_scoring
import diffusion_sources.temporal_early_baseline_runner as _early_runner
importlib.invalidate_caches()
importlib.reload(_early_scoring)
importlib.reload(_early_runner)
from diffusion_sources.temporal_early_baseline_runner import run_seed, summarize_seeds
from diffusion_sources.temporal_pilot_artifacts import write_stage, read_stage, sha256_file
BASELINE_DIR = DRIVE_ROOT / 'reports/runs/temporal_v3_early_baseline/v1'
BASELINE_RESULTS = {}
print('Saved pilot:', PILOT_DIR)
print('New baseline output:', BASELINE_DIR)
print('Device:', DEVICE, 'Готово. Следующая ячейка — только seed 7026.', flush=True)

## 9. Сравнение — frozen seed 7026

Replay и inference на validation, без обучения. Контрольный и temporal F1 и early-mask hash должны совпасть с прежним пилотом, иначе выполнение остановится.

In [ ]:
BASELINE_RESULTS[7026] = run_seed(DATA_DIR, RUN_DIR, PILOT_DIR, BASELINE_DIR, torch.device(DEVICE))

## 10. Сравнение — frozen seed 7027

Тот же ранний протокол, без нового подбора beta.

In [ ]:
BASELINE_RESULTS[7027] = run_seed(DATA_DIR, RUN_ROOT / 'frozen_candidate/seed_7027', PILOT_DIR, BASELINE_DIR, torch.device(DEVICE))

## 11. Сравнение — frozen seed 7028

In [ ]:
BASELINE_RESULTS[7028] = run_seed(DATA_DIR, RUN_ROOT / 'frozen_candidate/seed_7028', PILOT_DIR, BASELINE_DIR, torch.device(DEVICE))

## 12. Общий результат

Положительная разница temporal_minus_early означает вклад learned ranking сверх early membership при одинаковом count. CI — по каскадам того же validation, усреднение ties аналитическое. Это exploratory абляция; нет автоматического решения об обучении или доступе к закрытым данным.

In [ ]:
reports = [BASELINE_RESULTS[seed] for seed in (7026, 7027, 7028)]
summary = summarize_seeds(reports)
identity = {'seed_payloads': {str(seed): sha256_file(BASELINE_DIR / f'seed_{seed}/payload.json') for seed in (7026, 7027, 7028)}}
if (BASELINE_DIR / 'summary').exists():
    _, summary = read_stage(BASELINE_DIR, 'summary', identity)
else:
    write_stage(BASELINE_DIR, 'summary', {'identity': identity}, summary)
print(json.dumps(summary, indent=2, ensure_ascii=False), flush=True)
print('Summary:', BASELINE_DIR / 'summary/payload.json', flush=True)